In [13]:
# Importar bibliotecas

import sys
from pathlib import Path

In [14]:
# Settar a raiz do projeto

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [15]:
# Importar dados do JSON

from application.configuration.read_configuration import read_configuration_JSON 
from application.database.read_database import read_database_JSON 

DATABASE_JSON      = read_database_JSON()
CONFIGURATION_JSON = read_configuration_JSON()

In [16]:
# Coletar dados do JSON de configuração

from infra.object_value.compilation import Compilation

compilation_key = Compilation[CONFIGURATION_JSON["compilation"].upper()]

try:
    compilation = Compilation[compilation_key.name]
except KeyError:
    raise ValueError(
        "There is no valid key 'compilation' on 'configuration.json'"
    )

In [17]:
# Instanciando dados por produto

from domain.product.product_class import ProductDataBuild
from domain.purchase.purchase_class import ProductPurchaseBuild
from domain.shipping.shipping_factory import ProductShippingFactory
from domain.tax.tax_factory import ProductTaxFactory
from domain.delivery.delivery_factory import ProductDeliveryFactory
from domain.summarize.summarize_class import ProductSummarizeBuild
from domain.compilation_data.compilation_factory import CompilationDataFactory

from domain.product.helper.identifier import getIdentifierData
from infra.helper.transformEnumValue import transformEnumValue

array_sumarize_data         = []
array_product_full_data     = []
array_product_data          = []
array_product_purchase_data = []
array_product_shipping_data = []
array_product_tax_data      = []
array_product_delivery_data = []

for code in DATABASE_JSON:
    product_info = DATABASE_JSON[code]

    product_data = (
        ProductDataBuild()
        .setFigureName(DATABASE_JSON[code]["Nome_Figure"])
        .setMFCLink(DATABASE_JSON[code]["MFC_Link"])
        .setBrand(DATABASE_JSON[code]["Marca"])
        .setProcuctLine(DATABASE_JSON[code]["Linha"])
        .setScale(DATABASE_JSON[code]["Escala"])
        .build()
    )

    product_purchase_data = (
        ProductPurchaseBuild()
            .setPurchasePlace(product_info["Compra"]["Local_da_Compra"])
            .setPurchaseCountry(product_info["Compra"]["Pais_Local"])
            .setPurchasePaymentDate(product_info["Compra"]["Data_do_Pagamento"])
            .setPurchaseCurrencyPrice(product_info["Compra"]["Custos"]["Preco"])
            .setPurchaseCurrencyServiceTax(product_info["Compra"]["Custos"]["Taxa"])
            .setPurchasePaymentMethod(product_info["Compra"]["Metodo_Pagamento"])
            .setPurchaseQuote()
            .setPurchasePrice()
            .build()
        )

    product_shipping_data = ProductShippingFactory().create(product_info)
    product_tax_data      = ProductTaxFactory().create(product_info)
    product_delivery_data = ProductDeliveryFactory().create(product_info)
    
    product_summarize_data = (
        ProductSummarizeBuild()
        .setStatus(product_shipping_data, product_tax_data, product_delivery_data)
        .build()
    )
    
    product_identifier = getIdentifierData(product_data, code)

    compilation_data = CompilationDataFactory().create(
        product_summarize_data, 
        product_data, 
        product_purchase_data, 
        product_shipping_data, 
        product_tax_data, 
        product_delivery_data,
        product_identifier,
        compilation_key,
    )

    product_shipping_data = transformEnumValue(product_shipping_data, "shippingStatus")
    product_tax_data      = transformEnumValue(product_tax_data, "taxStatus")
    product_delivery_data = transformEnumValue(product_delivery_data, "deliveryStatus")

    array_sumarize_data.append(compilation_data["product_summarize_data"])

    if compilation_key == Compilation.SIMPLE or compilation_key == Compilation.FULL:
        array_product_data.append(compilation_data["product_data"])
        array_product_purchase_data.append(compilation_data["product_purchase_data"])
        array_product_shipping_data.append(compilation_data["product_shipping_data"])
        array_product_tax_data.append(compilation_data["product_tax_data"])
        array_product_delivery_data.append(compilation_data["product_delivery_data"])

    if compilation_key == Compilation.COMPLETE or compilation_key == Compilation.FULL:
        array_product_full_data.append(compilation_data["product_full_data"])
    


In [18]:
# Criar tabs

from src.domain.create_tab.create_tab_factory import CreateTabFactory 

tabs = []

tab_summarize_data = CreateTabFactory().create(array_sumarize_data, "summarized_data")
tabs.append(tab_summarize_data)

if compilation_key == Compilation.SIMPLE or compilation_key == Compilation.FULL:
    tab_product_data          = CreateTabFactory().create(array_product_data,          "data")
    tab_product_purchase_data = CreateTabFactory().create(array_product_purchase_data, "purchase_data")
    tab_product_shipping_data = CreateTabFactory().create(array_product_shipping_data, "shipping_data")
    tab_product_tax_data      = CreateTabFactory().create(array_product_tax_data,      "tax_data")
    tab_product_delivery_data = CreateTabFactory().create(array_product_delivery_data, "delivery_data")

    tabs.extend([tab_product_data, tab_product_purchase_data, tab_product_shipping_data, tab_product_tax_data, tab_product_delivery_data])

if compilation_key == Compilation.COMPLETE or compilation_key == Compilation.FULL:
    tab_full_data = CreateTabFactory().create(array_product_full_data, "full_data")
    tabs.append(tab_full_data)


In [19]:
# Gerar relatório

from src.domain.create_report.excel_report_builder import ExcelReportBuilder

(
    ExcelReportBuilder()
    .addTabs(tabs)
    .build("relatorio.xlsx")
)
